# جمع بيانات نظام العمل السعودي والأسئلة الشائعة — النسخة النهائية المعتمدة
## Web Scraping + Cleaning + Final Saving for RAG Pipeline

هذا النوتبوك مسؤول فقط عن **مرحلة السحب وحفظ البيانات الخام النظيفة**، حتى لا نعيد عملية السحب في كل نوتبوك لاحق.

المخرجات النهائية التي تعتمدها بقية المراحل تحفظ داخل:

```text
saudi_labor_law_voice_agent_project/01_raw/
```

ويتم حفظ مصدرين:

- مواد نظام العمل السعودي من موقع وزارة الموارد البشرية.
- الأسئلة الشائعة FAQ مع فلاتر التصنيف من موقع الوزارة.

> شغّل هذا النوتبوك مرة واحدة فقط عند الحاجة لتحديث البيانات من الموقع. بعد ذلك، نوتبوك المعالجة ونوتبوك RAG يقرؤون ملفات CSV المحفوظة مباشرة بدون إعادة Scraping.


## المرحلة 1: سحب مواد نظام العمل

تستخرج كل مادة مع نصها ورقمها وبابها وفصلها.

تتضمن محوّلاً يحوّل ترتيب المادة العربي إلى رقم صحيح، وتنتج المتغير df_labor_law.

In [1]:
# =========================================================
# Scrape Saudi Labor Law from HRSD by باب / فصل / مادة
# Source: HRSD - نظام العمل
# =========================================================

# إذا أول مرة تشغل الكود:
# !pip install requests beautifulsoup4 pandas openpyxl lxml

import re
import time
import json
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin


# ============================================================
# محوّل ترتيب المادة العربي إلى رقم
# ============================================================

ORDINAL_UNITS = {
    "الأولى":1,"الحادية":1,
    "الثانية":2,"الثالثة":3,"الرابعة":4,"الخامسة":5,
    "السادسة":6,"السابعة":7,"الثامنة":8,"التاسعة":9,"العاشرة":10,
}
ORDINAL_TENS = {
    "العشرون":20,"العشرين":20,"الثلاثون":30,"الثلاثين":30,
    "الأربعون":40,"الأربعين":40,"الخمسون":50,"الخمسين":50,
    "الستون":60,"الستين":60,"السبعون":70,"السبعين":70,
    "الثمانون":80,"الثمانين":80,"التسعون":90,"التسعين":90,
}


def _wrap_ordinal(n, bis):
    return f"{n}مكرر" if bis else str(n)


def arabic_ordinal_to_int(text):
    """تحويل ترتيب المادة العربي إلى نص رقمي، مع حفظ علامة (مكرر)."""
    if text is None:
        return None
    s = str(text).strip()
    if not s:
        return None

    bis = "مكرر" in s
    if bis:
        s = s.replace("مكرر", "").strip()

    hundreds = 0
    if "بعد المائتين" in s:
        hundreds = 200
        s = s.replace("بعد المائتين", "").strip()
    elif "بعد المائة" in s:
        hundreds = 100
        s = s.replace("بعد المائة", "").strip()

    s = re.sub(r"\s+", " ", s).strip()

    if s == "المائة":
        return _wrap_ordinal(100, bis)
    if s in ("المئتان", "المئتين"):
        return _wrap_ordinal(200, bis)
    if s == "" and hundreds:
        return _wrap_ordinal(hundreds, bis)

    base = None

    m = re.match(r"^(\S+)\s+عشرة$", s)
    if m and m.group(1) in ORDINAL_UNITS and ORDINAL_UNITS[m.group(1)] <= 9:
        base = 10 + ORDINAL_UNITS[m.group(1)]

    if base is None and " و" in s:
        ones_w, rest = s.split(" و", 1)
        ones_w = ones_w.strip()
        rest = rest.strip()
        tens_val = ORDINAL_TENS.get(rest) or ORDINAL_TENS.get("ال" + rest.lstrip("ال"))
        if ones_w in ORDINAL_UNITS and tens_val:
            base = ORDINAL_UNITS[ones_w] + tens_val

    if base is None and s in ORDINAL_TENS:
        base = ORDINAL_TENS[s]

    if base is None and s in ORDINAL_UNITS:
        base = ORDINAL_UNITS[s]

    if base is None:
        return None

    return _wrap_ordinal(base + hundreds, bis)


def article_number_to_int(label):
    """يحوّل ناتج المحوّل النصي إلى رقم صحيح، ويتجاهل (مكرر)."""
    if not label:
        return None
    digits = re.sub(r"\D", "", label)
    return int(digits) if digits else None


# ============================================================
# إعدادات السحب
# ============================================================

MAIN_URL = "https://www.hrsd.gov.sa/knowledge-centre/%D9%86%D8%B8%D8%A7%D9%85-%D8%A7%D9%84%D8%B9%D9%85%D9%84"
BASE_URL = "https://www.hrsd.gov.sa"

LAW_OUTPUT_CSV = "saudi_labor_law_by_bab_articles.csv"
LAW_OUTPUT_XLSX = "saudi_labor_law_by_bab_articles.xlsx"
LAW_OUTPUT_JSON = "saudi_labor_law_by_bab_articles.json"


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ar,en;q=0.9",
}


def clean_text(text: str) -> str:
    """تنظيف المسافات والرموز غير المرغوبة مع الحفاظ على العربية."""
    if text is None:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_soup(url: str) -> BeautifulSoup:
    """تحميل الصفحة وتحويلها إلى BeautifulSoup."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "lxml")


def remove_unwanted_tags(soup: BeautifulSoup) -> BeautifulSoup:
    """إزالة العناصر غير المفيدة مثل النافبار والفوتر والسكريبتات."""
    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    for selector in ["header", "footer", "nav"]:
        for tag in soup.select(selector):
            tag.decompose()

    return soup


def extract_bab_links(main_url: str) -> list:
    """
    استخراج روابط الأبواب من صفحة نظام العمل الرئيسية.
    """
    soup = get_soup(main_url)
    soup = remove_unwanted_tags(soup)

    h1 = None
    for candidate in soup.find_all("h1"):
        if "نظام العمل" in clean_text(candidate.get_text(" ")):
            h1 = candidate
            break

    if h1 is None:
        raise ValueError("لم أجد عنوان صفحة نظام العمل. قد تكون بنية الصفحة تغيّرت.")

    bab_links = []
    bab_order = 0

    for h in h1.find_all_next(["h2", "h3", "h4"]):
        heading_text = clean_text(h.get_text(" "))

        if "تاريخ آخر تحديث" in heading_text or "هل كانت هذه الصفحة مفيدة" in heading_text:
            break

        if re.match(r"^الباب\s+", heading_text):
            bab_order += 1
            bab_label = heading_text.replace(":", "").strip()

            link = h.find_next("a")
            if not link:
                continue

            bab_title = clean_text(link.get_text(" "))
            bab_url = urljoin(BASE_URL, link.get("href"))

            bab_links.append({
                "bab_order": bab_order,
                "bab_label": bab_label,
                "bab_title": bab_title,
                "bab_url": bab_url
            })

    return bab_links


def extract_article_number(article_title: str) -> str:
    """
    استخراج اسم المادة العربي من عنوان مثل:
    المادة الأولى:
    المادة الحادية عشرة مكرر :
    """
    article_title = clean_text(article_title)
    article_title = article_title.replace(":", "").strip()
    article_title = re.sub(r"^المادة\s*", "", article_title).strip()
    return article_title


def extract_articles_from_bab(bab: dict) -> list:
    """
    الدخول على صفحة باب واحد واستخراج:
    الباب، الفصل، المادة، نص المادة.
    """
    soup = get_soup(bab["bab_url"])
    soup = remove_unwanted_tags(soup)

    h1 = soup.find("h1")
    if h1 is None:
        raise ValueError(f"لم أجد عنوان الصفحة داخل الرابط: {bab['bab_url']}")

    rows = []

    current_fasl = ""
    current_article_title = ""
    current_article_number = ""
    current_article_lines = []

    def flush_article():
        """حفظ المادة الحالية قبل الانتقال للمادة التالية."""
        nonlocal current_article_title, current_article_number, current_article_lines

        article_text = clean_text(" ".join(current_article_lines))

        if current_article_title and article_text:
            searchable_text = clean_text(
                f"{bab['bab_label']} - {bab['bab_title']} - "
                f"{current_fasl} - {current_article_title} - {article_text}"
            )

            article_label = arabic_ordinal_to_int(current_article_number)

            rows.append({
                "bab_order": bab["bab_order"],
                "bab_label": bab["bab_label"],
                "bab_title": bab["bab_title"],
                "fasl": current_fasl,
                "article_title": current_article_title,
                "article_number": current_article_number,
                "article_number_label": article_label,
                "article_number_int": article_number_to_int(article_label),
                "article_text": article_text,
                "source_url": bab["bab_url"],
                "searchable_text": searchable_text
            })

        current_article_title = ""
        current_article_number = ""
        current_article_lines = []

    content_tags = h1.find_all_next(["h2", "h3", "h4", "h5", "p", "li"])

    previous_text = None

    for tag in content_tags:
        text = clean_text(tag.get_text(" "))

        if not text:
            continue

        stop_markers = [
            "تمثل صفحة المستفيد",
            "تاريخ آخر تحديث",
            "هل كانت هذه الصفحة مفيدة",
            "من فضلك أخبرنا بالسبب",
            "جميع الحقوق محفوظة"
        ]

        if any(marker in text for marker in stop_markers):
            break

        if text == previous_text:
            continue
        previous_text = text

        if text == bab["bab_title"]:
            continue

        if re.match(r"^الفصل\s+", text):
            flush_article()
            current_fasl = text
            continue

        if re.match(r"^المادة\s+", text):
            flush_article()
            current_article_title = text
            current_article_number = extract_article_number(text)
            current_article_lines = []
            continue

        if current_article_title:
            current_article_lines.append(text)

    flush_article()

    return rows


def main():
    print("جاري استخراج روابط الأبواب...")
    bab_links = extract_bab_links(MAIN_URL)

    print(f"عدد الأبواب التي تم العثور عليها: {len(bab_links)}")
    for bab in bab_links:
        print(f"- {bab['bab_label']} | {bab['bab_title']}")

    all_rows = []

    print("\nجاري استخراج المواد من كل باب...")
    for bab in bab_links:
        print(f"سحب: {bab['bab_label']} - {bab['bab_title']}")

        try:
            rows = extract_articles_from_bab(bab)
            all_rows.extend(rows)
            print(f"  تم استخراج {len(rows)} مادة")
            time.sleep(1)  # احترام الموقع وعدم الضغط عليه

        except Exception as e:
            print(f"  خطأ في {bab['bab_title']}: {e}")

    df = pd.DataFrame(all_rows)

    columns = [
        "bab_order",
        "bab_label",
        "bab_title",
        "fasl",
        "article_title",
        "article_number",
        "article_number_label",
        "article_number_int",
        "article_text",
        "source_url",
        "searchable_text"
    ]

    df = df[columns]

    # ============================================================
    # تنظيف نهائي بعد السحب
    # الهدف: إزالة التكرارات الناتجة من بنية صفحة الويب
    # مع الاحتفاظ بكل مادة قانونية حقيقية، بما فيها المواد الملغاة
    # ============================================================

    def normalize_article_text_for_save(text):
        text = clean_text(text)
        text = re.sub(r"[ـ]+", "", text)          # إزالة التطويل من مثل: ملغــــاة
        text = re.sub(r"\s+", " ", text).strip()
        return text

    df["article_text"] = df["article_text"].fillna("").astype(str).apply(normalize_article_text_for_save)
    df["text_len"] = df["article_text"].str.len()

    # تصنيف حالة المادة: فعالة أو ملغاة
    df["article_status"] = df["article_text"].apply(
        lambda x: "repealed" if ("ملغاة" in x or "ملغى" in x) else "active"
    )

    # مفتاح آمن للتكرار:
    # نستخدم article_number_label حتى لا نحذف المواد المكررة قانونياً لو ظهر نص "مكرر"
    # وإذا لم يوجد رقم واضح نستخدم عنوان المادة كحل احتياطي.
    df["article_unique_key"] = df["article_number_label"].fillna("").astype(str).str.strip()
    missing_key = df["article_unique_key"].isin(["", "None", "nan", "NaN"])
    df.loc[missing_key, "article_unique_key"] = df.loc[missing_key, "article_title"].fillna("").astype(str).str.strip()

    rows_before = len(df)

    # إزالة التكرار مع الاحتفاظ بأطول نص لكل مادة
    df = (
        df.sort_values(["article_unique_key", "text_len"], ascending=[True, False])
          .drop_duplicates(subset=["article_unique_key"], keep="first")
          .sort_values(["article_number_int", "article_title"], ascending=[True, True])
          .reset_index(drop=True)
    )

    rows_after = len(df)

    # تحديث searchable_text بعد التنظيف
    df["searchable_text"] = df.apply(
        lambda row: clean_text(
            f"{row['bab_label']} - {row['bab_title']} - "
            f"{row['fasl']} - {row['article_title']} - {row['article_text']}"
        ),
        axis=1
    )

    # حذف المفتاح المساعد من ملف الحفظ
    df = df.drop(columns=["article_unique_key"])

    # فحص جودة نهائي
    nums = set(df["article_number_int"].dropna().astype(int))
    missing_articles = [n for n in range(1, 246) if n not in nums]
    duplicated_labels = df["article_number_label"].duplicated().sum()
    empty_texts = (df["article_text"].astype(str).str.strip() == "").sum()

    print("\nفحص جودة مواد نظام العمل بعد التنظيف:")
    print(f"عدد الصفوف قبل إزالة التكرار: {rows_before}")
    print(f"عدد الصفوف بعد إزالة التكرار: {rows_after}")
    print(f"عدد المواد: {len(nums)} | الأعلى: {max(nums) if nums else None}")
    print(f"المفقود من 1 إلى 245: {missing_articles}")
    print(f"عدد التكرارات حسب رقم المادة: {duplicated_labels}")
    print(f"عدد المواد بدون نص: {empty_texts}")
    print("توزيع حالة المواد:")
    print(df["article_status"].value_counts())

    df.to_csv(LAW_OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df.to_excel(LAW_OUTPUT_XLSX, index=False)

    with open(LAW_OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

    print("\nتم الانتهاء بنجاح")
    print(f"عدد المواد النهائي: {len(df)}")
    print(f"CSV  : {LAW_OUTPUT_CSV}")
    print(f"Excel: {LAW_OUTPUT_XLSX}")
    print(f"JSON : {LAW_OUTPUT_JSON}")

    return df


df_labor_law = main()

df_labor_law.head(10)

جاري استخراج روابط الأبواب...
عدد الأبواب التي تم العثور عليها: 16
- الباب الأول | التعريفات / الأحكام العامة
- الباب الثاني | تنظيم عمليات التوظيف
- الباب الثالث | توظيف غير السعوديين
- الباب الرابع | التدريب والتأهيل
- الباب الخامس | علاقات العمل
- الباب السادس | شروط العمل وظروفه
- الباب السابع | العمل لبعض الوقت
- الباب الثامن | الوقاية من مخاطر العمل والوقاية من الحوادث الصناعية الكبرى وإصابات العمل والخدمات الصحية والاجتماعية
- الباب التاسع | تشغيل النساء
- الباب العاشر | تشغيل الأحداث
- الباب الحادي عشر | عقد العمل البحري
- الباب الثاني عشر | العمل في المناجم والمحاجر
- الباب الثالث عشر | تفتيش العمل
- الباب الرابع عشر | هيئات تسوية الخلافات العمالية
- الباب الخامس عشر | العقوبات
- الباب السادس عشر | أحكام ختامية

جاري استخراج المواد من كل باب...
سحب: الباب الأول - التعريفات / الأحكام العامة
  تم استخراج 22 مادة
سحب: الباب الثاني - تنظيم عمليات التوظيف
  تم استخراج 10 مادة
سحب: الباب الثالث - توظيف غير السعوديين
  تم استخراج 10 مادة
سحب: الباب الرابع - التدريب والتأهيل
  تم استخ

,bab_order,bab_label,bab_title,fasl,article_title,article_number,article_number_label,article_number_int,article_text,source_url,searchable_text,text_len,article_status
0,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الأول,المادة الأولى:,الأولى,1,1,يسمى هذا النظام نظام العمل.,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,27,active
1,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الأول,المادة الثانية :,الثانية,2,2,يقصد بالألفاظ والعبارات الآتية – أينما وردت في...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,3519,active
2,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الثالثة:,الثالثة,3,3,العمل حق للمواطن ، لا يجوز لغيره ممارسته إلا ب...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,258,active
3,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الرابعة:,الرابعة,4,4,يجب على صاحب العمل والعامل عند تطبيق أحكام هذا...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,97,active
4,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الخامسة:,الخامسة,5,5,تسري أحكام هذا النظام على الآتي: كل عقد عمل يل...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,433,active
5,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة السادسة:,السادسة,6,6,تسري على العامل العرضي والموسمي والمؤقت الأحكا...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,272,active
6,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة السابعة :,السابعة,7,7,1- يستثنى من تطبيق أحكام هذا النظام كل من: أ‌-...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,1030,active
7,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الثامنة:,الثامنة,8,8,يبطل كل شرط يخالف أحكام هذا النظام ، ويبطل كل ...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,160,active
8,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة التاسعة:,التاسعة,9,9,اللغة العربية هي الواجبة الإستعمال في البيانات...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,325,active
9,1,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة العاشرة:,العاشرة,10,10,تحسب جميع المدد والمواعيد المنصوص عليها في هذا...,https://www.hrsd.gov.sa/node/5575978,الباب الأول - التعريفات / الأحكام العامة - الف...,129,active


### 1.1 معاينة المواد المسحوبة

In [2]:
df_labor_law[["article_title", "article_number", "article_number_int"]].head(15)

,article_title,article_number,article_number_int
0,المادة الأولى:,الأولى,1
1,المادة الثانية :,الثانية,2
2,المادة الثالثة:,الثالثة,3
3,المادة الرابعة:,الرابعة,4
4,المادة الخامسة:,الخامسة,5
5,المادة السادسة:,السادسة,6
6,المادة السابعة :,السابعة,7
7,المادة الثامنة:,الثامنة,8
8,المادة التاسعة:,التاسعة,9
9,المادة العاشرة:,العاشرة,10


### 1.2 فحص اكتمال المواد

يتحقق من تغطية الأرقام من 1 إلى 245. يجب أن تكون قائمة المفقود فارغة.

In [3]:
nums = set(df_labor_law["article_number_int"].dropna().astype(int))
print("عدد المواد:", len(nums), "| الأعلى:", max(nums))
print("المفقود من 1 إلى 245:", [n for n in range(1, 246) if n not in nums])

عدد المواد: 245 | الأعلى: 245
المفقود من 1 إلى 245: []


### 1.3 تشخيص جودة المواد

يعدّ التكرارات، والمواد بلا نص، والنصوص القصيرة جداً، ويعرض أمثلة عليها.

ملاحظة: التكرارات الأربعة هي مواد (مكرر) التي تشترك في الرقم الأساسي، وليست تكراراً حقيقياً.

In [4]:
# ============================================================
# تشخيص التكرارات والمواد القصيرة
# ============================================================

df_check = df_labor_law.copy()

df_check["article_text"] = df_check["article_text"].fillna("").astype(str).str.strip()
df_check["text_len"] = df_check["article_text"].str.len()

print("عدد الصفوف الكلي:", len(df_check))
print("عدد أرقام المواد الفريدة:", df_check["article_number_int"].nunique())
print("عدد التكرارات في رقم المادة:", df_check["article_number_int"].duplicated().sum())
print("عدد المواد بدون نص:", (df_check["article_text"] == "").sum())
print("عدد المواد بنص قصير جداً:", (df_check["text_len"] < 30).sum())

print("\n=== المواد المكررة ===")
duplicated_articles = (
    df_check[df_check["article_number_int"].duplicated(keep=False)]
    .sort_values(["article_number_int", "text_len"], ascending=[True, False])
)

display(
    duplicated_articles[
        ["article_number_int", "article_title", "article_text", "text_len"]
    ]
)

print("\n=== المواد ذات النص القصير جداً ===")
short_articles = (
    df_check[df_check["text_len"] < 30]
    .sort_values(["article_number_int"])
)

display(
    short_articles[
        ["article_number_int", "article_title", "article_text", "text_len"]
    ]
)

عدد الصفوف الكلي: 249
عدد أرقام المواد الفريدة: 245
عدد التكرارات في رقم المادة: 4
عدد المواد بدون نص: 0
عدد المواد بنص قصير جداً: 39

=== المواد المكررة ===


,article_number_int,article_title,article_text,text_len
11,11,المادة الحادية عشرة:,إذا عهد صاحب العمل لأي شخص طبيعي أو معنوي القي...,176
10,11,المادة الحادية عشرة مكرر :,مع عدم الإخلال بأحكام هذا النظام والأنظمة ذات ...,160
79,79,المادة التاسعة والسبعون مكرر:,يُعد طلب الاستقالة المقدم مقبولاً إذا مضى على ...,930
80,79,المادة التاسعة والسبعون:,لا ينقضي عقد العمل بوفاة صاحب العمل ، ما لم تك...,225
132,131,المادة الحادية والثلاثون بعد المائة مكرر:,يحدد الوزير - بقرار منه - المهن والأعمال التي ...,341
133,131,المادة الحادية والثلاثون بعد المائة:,يصدر الوزير اللوائح والقرارات التي تتضمن الترت...,340
231,229,المادة التاسعة والعشرون بعد المائتين :,1 – مع عدم الإخلال بأي عقوبة أشد ينص عليها نظا...,427
232,229,المادة التاسعة والعشرون بعد المائتين مكرر:,"يعاقب بغرامة لا تقل عن (200,000) مائتي ألف ريا...",224



=== المواد ذات النص القصير جداً ===


,article_number_int,article_title,article_text,text_len
0,1,المادة الأولى:,يسمى هذا النظام نظام العمل.,27
14,14,المادة الرابعة عشرة:,(ملغاة),7
151,149,المادة التاسعة والأربعون بعد المائة:,(ملغاة),7
152,150,المادة الخمسون بعد المائة:,(ملغاة),7
154,152,المادة الثانية والخمسون بعد المائة:,(ملغاة),7
158,156,المادة السادسة والخمسون بعد المائة:,(ملغاة),7
197,195,المادة الخامسة والتسعون بعد المائة:,(ملغاة),7
199,197,المادة السابعة والتسعون بعد المائة:,(ملغاة),7
205,203,المادة الثالثة بعد المائتين:,(ملغاة),7
207,205,المادة الخامسة بعد المائتين:,(ملغاة),7


### 1.4 تصنيف الحالة وإزالة التكرار النهائية

تصنّف كل مادة إلى:

- `active`: مادة فعّالة.
- `repealed`: مادة ملغاة.

كما يتم إزالة التكرار الحقيقي على أساس `article_number_label` وليس `article_number_int` فقط، لأن بعض مواد النظام تكون قانونياً بصيغة **مكرر**، وهذه يجب الحفاظ عليها وعدم اعتبارها تكراراً خاطئاً.


In [5]:
# ============================================================
# Final Law Cleaning Cell
# تصنيف حالة المادة + إزالة التكرار الحقيقي مع الحفاظ على مواد (مكرر)
# ============================================================

import re
import json

# أسماء ملفات مواد نظام العمل فقط
LAW_OUTPUT_CSV = "saudi_labor_law_by_bab_articles.csv"
LAW_OUTPUT_XLSX = "saudi_labor_law_by_bab_articles.xlsx"
LAW_OUTPUT_JSON = "saudi_labor_law_by_bab_articles.json"


def normalize_arabic_text(text):
    """تنظيف المسافات والتطويل مع الحفاظ على النص العربي."""
    text = str(text)
    text = re.sub(r"[ـ]+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def detect_article_status(text):
    """تصنيف المادة إلى فعالة أو ملغاة."""
    normalized = normalize_arabic_text(text)
    return "repealed" if ("ملغاة" in normalized or "ملغى" in normalized) else "active"


# نسخة تنظيف نهائية
df_labor_law_clean = df_labor_law.copy()

# تنظيف النصوص
for col in ["bab_label", "bab_title", "fasl", "article_title", "article_text"]:
    if col in df_labor_law_clean.columns:
        df_labor_law_clean[col] = (
            df_labor_law_clean[col]
            .fillna("")
            .astype(str)
            .apply(normalize_arabic_text)
        )

# قياسات ومؤشرات
df_labor_law_clean["text_len"] = df_labor_law_clean["article_text"].str.len()
df_labor_law_clean["article_status"] = df_labor_law_clean["article_text"].apply(detect_article_status)

# مفتاح إزالة التكرار الحقيقي:
# article_number_label يحافظ على مواد مثل 74مكرر ولا يخلطها مع 74
if "article_number_label" not in df_labor_law_clean.columns:
    df_labor_law_clean["article_number_label"] = df_labor_law_clean["article_number_int"].astype(str)

df_labor_law_clean["article_unique_key"] = (
    df_labor_law_clean["article_number_label"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# احتياطياً لو كانت المادة بلا label واضح
missing_key = df_labor_law_clean["article_unique_key"].isin(["", "None", "nan", "NaN"])
df_labor_law_clean.loc[missing_key, "article_unique_key"] = (
    df_labor_law_clean.loc[missing_key, "article_title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

rows_before = len(df_labor_law_clean)

# إزالة التكرارات الحقيقية مع الاحتفاظ بأطول نص لكل مادة
df_labor_law_clean = (
    df_labor_law_clean
    .sort_values(["article_unique_key", "text_len"], ascending=[True, False])
    .drop_duplicates(subset=["article_unique_key"], keep="first")
    .sort_values(["article_number_int", "article_number_label"], ascending=[True, True])
    .reset_index(drop=True)
)

rows_after = len(df_labor_law_clean)

# تحديث searchable_text بعد التنظيف
if set(["bab_label", "bab_title", "fasl", "article_title", "article_text"]).issubset(df_labor_law_clean.columns):
    df_labor_law_clean["searchable_text"] = df_labor_law_clean.apply(
        lambda row: normalize_arabic_text(
            f"{row['bab_label']} - {row['bab_title']} - {row['fasl']} - "
            f"{row['article_title']} - {row['article_text']}"
        ),
        axis=1
    )

# حذف المفتاح المساعد من الملف النهائي
df_labor_law_clean = df_labor_law_clean.drop(columns=["article_unique_key"], errors="ignore")

# فحص نهائي
nums = set(df_labor_law_clean["article_number_int"].dropna().astype(int))
missing_articles = [n for n in range(1, 246) if n not in nums]
true_duplicates = df_labor_law_clean["article_number_label"].duplicated().sum()
legal_repeated_numbers = df_labor_law_clean["article_number_int"].duplicated().sum()
empty_texts = (df_labor_law_clean["article_text"].astype(str).str.strip() == "").sum()

print("عدد الصفوف قبل إزالة التكرار:", rows_before)
print("عدد الصفوف بعد إزالة التكرار:", rows_after)
print("عدد المواد:", len(nums), "| الأعلى:", max(nums) if nums else None)
print("المفقود من 1 إلى 245:", missing_articles)
print("عدد التكرارات الحقيقية حسب article_number_label:", true_duplicates)
print("عدد مواد مكرر القانونية حسب article_number_int:", legal_repeated_numbers)
print("عدد المواد بدون نص:", empty_texts)

print("\nتوزيع حالة المواد:")
display(df_labor_law_clean["article_status"].value_counts().to_frame("count"))

print("\nالمواد القصيرة غير الملغاة:")
display(
    df_labor_law_clean[
        (df_labor_law_clean["text_len"] < 30) &
        (df_labor_law_clean["article_status"] != "repealed")
    ][["article_number_int", "article_number_label", "article_title", "article_text", "text_len", "article_status"]]
)

# حفظ نسخة قانونية مباشرة في نفس مجلد النوتبوك
# وسيتم أيضاً حفظ نسخة موحدة داخل مجلد المشروع في آخر النوتبوك
df_labor_law_clean.to_csv(LAW_OUTPUT_CSV, index=False, encoding="utf-8-sig")
df_labor_law_clean.to_excel(LAW_OUTPUT_XLSX, index=False)

with open(LAW_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(df_labor_law_clean.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

print("\nتم حفظ مواد نظام العمل النظيفة:")
print(LAW_OUTPUT_CSV)
print(LAW_OUTPUT_XLSX)
print(LAW_OUTPUT_JSON)


عدد الصفوف قبل إزالة التكرار: 249
عدد الصفوف بعد إزالة التكرار: 249
عدد المواد: 245 | الأعلى: 245
المفقود من 1 إلى 245: []
عدد التكرارات الحقيقية حسب article_number_label: 0
عدد مواد مكرر القانونية حسب article_number_int: 4
عدد المواد بدون نص: 0

توزيع حالة المواد:


,count
article_status,
active,211
repealed,38



المواد القصيرة غير الملغاة:


,article_number_int,article_number_label,article_title,article_text,text_len,article_status
0,1,1,المادة الأولى:,يسمى هذا النظام نظام العمل.,27,active



تم حفظ مواد نظام العمل النظيفة:
saudi_labor_law_by_bab_articles.csv
saudi_labor_law_by_bab_articles.xlsx
saudi_labor_law_by_bab_articles.json


## المرحلة 2: سحب الأسئلة الشائعة

تفتح متصفحاً وتسحب كل الأسئلة الشائعة وتصنّفها حسب فلاتر الموقع، وتنتج المتغير df_faq.

قد تأخذ وقتاً لأنها تمر على عشرات الصفحات وعدة فلاتر.

In [6]:
# =========================================================
# HRSD FAQ Scraper with Filters as Classification Columns
# يسحب الأسئلة + الأجوبة + تصنيف المستفيدين + القطاع + الموقع الفرعي
# =========================================================

# أول مرة فقط:
# !pip install selenium beautifulsoup4 pandas openpyxl lxml

import re
import time
import json
import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.common.exceptions import NoSuchElementException, TimeoutException


BASE_URL = "https://www.hrsd.gov.sa/contact-us/faq"
TOTAL_PAGES = 60

FAQ_OUTPUT_CSV = "hrsd_faq_with_filters_classification.csv"
FAQ_OUTPUT_XLSX = "hrsd_faq_with_filters_classification.xlsx"
FAQ_OUTPUT_JSON = "hrsd_faq_with_filters_classification.json"


# ---------------------------------------------------------
# Basic Helpers
# ---------------------------------------------------------

def clean_text(text):
    if text is None:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_key(question, answer):
    """
    مفتاح ثابت لمطابقة نفس السؤال والجواب حتى لو كان فيه اختلاف مسافات.
    """
    q = clean_text(question)
    a = clean_text(answer)
    return f"{q}|||{a}"


def setup_driver(headless=False):
    chrome_options = Options()

    if headless:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--lang=ar-SA")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=chrome_options)
    return driver


def wait_page_ready(driver, seconds=30):
    WebDriverWait(driver, seconds).until(
        lambda d: "الأسئلة الشائعة" in d.page_source
    )
    time.sleep(1.5)


# ---------------------------------------------------------
# Extract FAQ from current HTML
# ---------------------------------------------------------

def extract_faqs_from_html(html, page_number, current_url):
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    h1 = None
    for candidate in soup.find_all("h1"):
        if "الأسئلة الشائعة" in clean_text(candidate.get_text(" ")):
            h1 = candidate
            break

    if h1 is None:
        return []

    rows = []

    current_question = None
    current_answer_parts = []

    def flush():
        nonlocal current_question, current_answer_parts

        answer = clean_text(" ".join(current_answer_parts))

        if current_question and answer:
            rows.append({
                "question": current_question,
                "answer": answer,
                "page_number": page_number,
                "page_url": current_url,
                "source": "HRSD FAQ",
                "searchable_text": clean_text(
                    f"السؤال: {current_question} الإجابة: {answer}"
                )
            })

        current_question = None
        current_answer_parts = []

    stop_markers = [
        "Pagination",
        "تمثل صفحة المستفيد",
        "تاريخ آخر تحديث",
        "هل كانت هذه الصفحة مفيدة",
        "جميع الحقوق محفوظة",
        "تابعنا على",
        "تطبيقات الوزارة"
    ]

    ignore_lines = {
        "العنوان",
        "المستفيدين",
        "القطاع",
        "الموقع الفرعي",
        "ترتيب حسب",
        "- اختر -",
        "أحدث",
        "الأقدم",
        "العنوان (أ - ي)",
        "العنوان (ي - أ)",
        "Next page",
        "Previous page",
        "…",
    }

    # لا نستخدم div حتى لا يتكرر كامل محتوى الصفحة
    tags = h1.find_all_next(["h2", "h3", "h4", "p", "li"])

    for tag in tags:
        text = clean_text(tag.get_text(" "))

        if not text:
            continue

        if text in ignore_lines:
            continue

        if re.fullmatch(r"Page\s+\d+", text, re.IGNORECASE):
            continue

        if any(marker in text for marker in stop_markers):
            flush()
            break

        # السؤال غالباً h2
        if tag.name == "h2":
            flush()
            current_question = text
            current_answer_parts = []
            continue

        # الجواب بعد السؤال
        if current_question and tag.name in ["p", "li"]:
            current_answer_parts.append(text)

    flush()
    return rows


def get_first_question(driver):
    rows = extract_faqs_from_html(driver.page_source, page_number=0, current_url=driver.current_url)
    if not rows:
        return ""
    return rows[0]["question"]


# ---------------------------------------------------------
# Pagination
# ---------------------------------------------------------

def click_next_page(driver):
    """
    الضغط على Next page فعلياً.
    """

    xpaths = [
        "//a[@rel='next']",
        "//a[contains(@aria-label, 'Next page')]",
        "//a[contains(@title, 'Next page')]",
        "//a[contains(normalize-space(.), 'Next page')]",
        "//button[contains(normalize-space(.), 'Next page')]",
        "//a[contains(normalize-space(.), 'التالي')]",
        "//button[contains(normalize-space(.), 'التالي')]",
    ]

    next_element = None

    for xp in xpaths:
        elements = driver.find_elements(By.XPATH, xp)
        elements = [e for e in elements if e.is_displayed() and e.is_enabled()]
        if elements:
            next_element = elements[-1]
            break

    if next_element is None:
        return False

    driver.execute_script(
        "arguments[0].scrollIntoView({block: 'center'});",
        next_element
    )
    time.sleep(0.5)

    old_first = get_first_question(driver)

    driver.execute_script("arguments[0].click();", next_element)

    try:
        WebDriverWait(driver, 20).until(
            lambda d: get_first_question(d) != old_first
        )
    except:
        time.sleep(3)

    return True


# ---------------------------------------------------------
# Filter Options Extraction
# ---------------------------------------------------------

def get_visible_lines_from_html(html):
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    text = soup.get_text("\n")
    lines = [clean_text(x) for x in text.split("\n")]
    lines = [x for x in lines if x]

    return lines


def extract_options_between(lines, start_label, end_label):
    """
    استخراج خيارات الفلتر من النص الظاهر:
    مثلا بين "المستفيدين" و "القطاع"
    """
    if start_label not in lines:
        return []

    start_idx = lines.index(start_label) + 1

    if end_label in lines:
        end_idx = lines.index(end_label)
    else:
        end_idx = len(lines)

    options = lines[start_idx:end_idx]

    options = [
        x for x in options
        if x not in ["- اختر -", "", "العنوان"]
    ]

    # إزالة التكرار مع الحفاظ على الترتيب
    seen = set()
    clean_options = []

    for opt in options:
        if opt not in seen:
            seen.add(opt)
            clean_options.append(opt)

    return clean_options


def get_filter_options(driver):
    """
    استخراج خيارات الفلاتر الموجودة في الصفحة.
    """
    lines = get_visible_lines_from_html(driver.page_source)

    beneficiaries = extract_options_between(lines, "المستفيدين", "القطاع")
    sectors = extract_options_between(lines, "القطاع", "الموقع الفرعي")
    subsites = extract_options_between(lines, "الموقع الفرعي", "ترتيب حسب")
    sort_options = extract_options_between(lines, "ترتيب حسب", get_first_question(driver))

    return {
        "beneficiaries": beneficiaries,
        "sectors": sectors,
        "subsites": subsites,
        "sort_options": sort_options,
    }


# ---------------------------------------------------------
# Apply Filter
# ---------------------------------------------------------

def try_select_native_dropdown(driver, label_text, option_text):
    """
    يحاول اختيار الفلتر إذا كان HTML select عادي.
    """
    try:
        select_el = driver.find_element(
            By.XPATH,
            f"//*[normalize-space()='{label_text}']/following::select[1]"
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            select_el
        )
        time.sleep(0.3)

        Select(select_el).select_by_visible_text(option_text)

        driver.execute_script(
            """
            arguments[0].dispatchEvent(new Event('change', {bubbles: true}));
            arguments[0].dispatchEvent(new Event('input', {bubbles: true}));
            """,
            select_el
        )

        return True

    except Exception:
        return False


def try_select_custom_dropdown(driver, label_text, option_text):
    """
    يحاول اختيار الفلتر إذا كان Dropdown مخصص وليس select.
    """
    try:
        # أقرب عنصر قابل للضغط بعد عنوان الفلتر
        dropdown = driver.find_element(
            By.XPATH,
            f"//*[normalize-space()='{label_text}']/following::*[self::button or @role='combobox' or contains(@class,'select')][1]"
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            dropdown
        )
        time.sleep(0.5)

        driver.execute_script("arguments[0].click();", dropdown)
        time.sleep(0.7)

        option = driver.find_element(
            By.XPATH,
            f"//*[normalize-space()='{option_text}']"
        )

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            option
        )
        time.sleep(0.3)

        driver.execute_script("arguments[0].click();", option)
        return True

    except Exception:
        return False


def click_apply_or_wait(driver):
    """
    بعض الصفحات تطبق الفلتر تلقائياً، وبعضها تحتاج زر بحث/تطبيق.
    """
    possible_buttons = [
        "//button[contains(normalize-space(.), 'تطبيق')]",
        "//input[@type='submit' and contains(@value, 'تطبيق')]",
        "//button[contains(normalize-space(.), 'بحث')]",
        "//input[@type='submit' and contains(@value, 'بحث')]",
        "//button[contains(normalize-space(.), 'Apply')]",
        "//input[@type='submit' and contains(@value, 'Apply')]",
    ]

    clicked = False

    for xp in possible_buttons:
        buttons = driver.find_elements(By.XPATH, xp)
        buttons = [b for b in buttons if b.is_displayed() and b.is_enabled()]
        if buttons:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                buttons[0]
            )
            time.sleep(0.3)
            driver.execute_script("arguments[0].click();", buttons[0])
            clicked = True
            break

    time.sleep(2.5)
    return clicked


def apply_filter(driver, label_text, option_text):
    """
    تطبيق فلتر واحد.
    """
    old_first = get_first_question(driver)

    ok = try_select_native_dropdown(driver, label_text, option_text)

    if not ok:
        ok = try_select_custom_dropdown(driver, label_text, option_text)

    if not ok:
        print(f"    لم أستطع اختيار الفلتر: {label_text} = {option_text}")
        return False

    click_apply_or_wait(driver)

    try:
        WebDriverWait(driver, 20).until(
            lambda d: get_first_question(d) != old_first or option_text in d.page_source
        )
    except:
        time.sleep(3)

    return True


# ---------------------------------------------------------
# Scrape current filtered results
# ---------------------------------------------------------

def scrape_current_results(driver, max_pages=60):
    """
    يسحب نتائج الوضع الحالي للصفحة، سواء بدون فلتر أو بفلتر.
    """
    rows = []
    seen_on_filter = set()

    for page_number in range(1, max_pages + 1):
        page_rows = extract_faqs_from_html(
            html=driver.page_source,
            page_number=page_number,
            current_url=driver.current_url
        )

        new_count = 0

        for row in page_rows:
            key = normalize_key(row["question"], row["answer"])
            if key not in seen_on_filter:
                seen_on_filter.add(key)
                rows.append(row)
                new_count += 1

        print(f"      صفحة {page_number}: المستخرج {len(page_rows)} | الجديد {new_count}")

        if page_number == max_pages:
            break

        clicked = click_next_page(driver)

        if not clicked:
            break

        time.sleep(1.2)

    return rows


# ---------------------------------------------------------
# Main Scraping
# ---------------------------------------------------------

def scrape_faq_with_filter_classification():
    driver = setup_driver(headless=False)

    try:
        # 1) سحب كل الأسئلة بدون فلتر
        print("فتح صفحة الأسئلة الشائعة...")
        driver.get(BASE_URL)
        wait_page_ready(driver)

        print("\nاستخراج خيارات الفلاتر...")
        filter_options = get_filter_options(driver)

        print("خيارات المستفيدين:", len(filter_options["beneficiaries"]))
        print("خيارات القطاع:", len(filter_options["sectors"]))
        print("خيارات الموقع الفرعي:", len(filter_options["subsites"]))

        print("\nسحب جميع الأسئلة بدون فلتر...")
        master_rows = scrape_current_results(driver, max_pages=TOTAL_PAGES)

        master_map = {}

        for row in master_rows:
            key = normalize_key(row["question"], row["answer"])
            master_map[key] = row
            master_map[key]["beneficiaries"] = set()
            master_map[key]["sector"] = set()
            master_map[key]["subsite"] = set()

        # 2) تصنيف حسب المستفيدين
        print("\nتصنيف الأسئلة حسب المستفيدين...")
        for option in filter_options["beneficiaries"]:
            print(f"  فلتر المستفيدين: {option}")

            driver.get(BASE_URL)
            wait_page_ready(driver)

            if not apply_filter(driver, "المستفيدين", option):
                continue

            rows = scrape_current_results(driver, max_pages=TOTAL_PAGES)

            for row in rows:
                key = normalize_key(row["question"], row["answer"])

                if key not in master_map:
                    master_map[key] = row
                    master_map[key]["beneficiaries"] = set()
                    master_map[key]["sector"] = set()
                    master_map[key]["subsite"] = set()

                master_map[key]["beneficiaries"].add(option)

        # 3) تصنيف حسب القطاع
        print("\nتصنيف الأسئلة حسب القطاع...")
        for option in filter_options["sectors"]:
            print(f"  فلتر القطاع: {option}")

            driver.get(BASE_URL)
            wait_page_ready(driver)

            if not apply_filter(driver, "القطاع", option):
                continue

            rows = scrape_current_results(driver, max_pages=TOTAL_PAGES)

            for row in rows:
                key = normalize_key(row["question"], row["answer"])

                if key not in master_map:
                    master_map[key] = row
                    master_map[key]["beneficiaries"] = set()
                    master_map[key]["sector"] = set()
                    master_map[key]["subsite"] = set()

                master_map[key]["sector"].add(option)

        # 4) تصنيف حسب الموقع الفرعي
        print("\nتصنيف الأسئلة حسب الموقع الفرعي...")
        for option in filter_options["subsites"]:
            print(f"  فلتر الموقع الفرعي: {option}")

            driver.get(BASE_URL)
            wait_page_ready(driver)

            if not apply_filter(driver, "الموقع الفرعي", option):
                continue

            rows = scrape_current_results(driver, max_pages=TOTAL_PAGES)

            for row in rows:
                key = normalize_key(row["question"], row["answer"])

                if key not in master_map:
                    master_map[key] = row
                    master_map[key]["beneficiaries"] = set()
                    master_map[key]["sector"] = set()
                    master_map[key]["subsite"] = set()

                master_map[key]["subsite"].add(option)

    finally:
        driver.quit()

    # تحويل الداتا النهائية إلى DataFrame
    final_rows = []

    for key, row in master_map.items():
        beneficiaries = " | ".join(sorted(row.get("beneficiaries", set())))
        sector = " | ".join(sorted(row.get("sector", set())))
        subsite = " | ".join(sorted(row.get("subsite", set())))

        question = row["question"]
        answer = row["answer"]

        final_rows.append({
            "question": question,
            "answer": answer,
            "beneficiaries": beneficiaries,
            "sector": sector,
            "subsite": subsite,
            "page_number": row.get("page_number", ""),
            "page_url": row.get("page_url", ""),
            "source": "HRSD FAQ",
            "searchable_text": clean_text(
                f"التصنيف: {beneficiaries} {sector} {subsite} "
                f"السؤال: {question} الإجابة: {answer}"
            )
        })

    df = pd.DataFrame(final_rows)

    if df.empty:
        return df, filter_options

    df = df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)
    df.insert(0, "faq_id", range(1, len(df) + 1))

    df = df[
        [
            "faq_id",
            "question",
            "answer",
            "beneficiaries",
            "sector",
            "subsite",
            "page_number",
            "page_url",
            "source",
            "searchable_text"
        ]
    ]

    return df, filter_options


df_faq, filter_options = scrape_faq_with_filter_classification()

print("\nتم الانتهاء")
print("عدد الأسئلة النهائي:", len(df_faq))

df_faq.head(20)

فتح صفحة الأسئلة الشائعة...

استخراج خيارات الفلاتر...
خيارات المستفيدين: 41
خيارات القطاع: 4
خيارات الموقع الفرعي: 3

سحب جميع الأسئلة بدون فلتر...
      صفحة 1: المستخرج 12 | الجديد 12
      صفحة 2: المستخرج 12 | الجديد 11
      صفحة 3: المستخرج 12 | الجديد 12
      صفحة 4: المستخرج 12 | الجديد 12
      صفحة 5: المستخرج 12 | الجديد 12
      صفحة 6: المستخرج 12 | الجديد 12
      صفحة 7: المستخرج 12 | الجديد 12
      صفحة 8: المستخرج 12 | الجديد 12
      صفحة 9: المستخرج 12 | الجديد 12
      صفحة 10: المستخرج 12 | الجديد 12
      صفحة 11: المستخرج 12 | الجديد 12
      صفحة 12: المستخرج 12 | الجديد 12
      صفحة 13: المستخرج 12 | الجديد 12
      صفحة 14: المستخرج 12 | الجديد 12
      صفحة 15: المستخرج 12 | الجديد 12
      صفحة 16: المستخرج 12 | الجديد 12
      صفحة 17: المستخرج 12 | الجديد 10
      صفحة 18: المستخرج 12 | الجديد 12
      صفحة 19: المستخرج 12 | الجديد 12
      صفحة 20: المستخرج 12 | الجديد 12
      صفحة 21: المستخرج 12 | الجديد 12
      صفحة 22: المستخرج 12 | الجديد 12
  

,faq_id,question,answer,beneficiaries,sector,subsite,page_number,page_url,source,searchable_text
0,1,هل تستخدم البوابة والانظمة التابعة لها الحلول ...,تقدم الوزارة خدمة التوقيع الالكتروني في اطار ا...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: أصحاب عمل | أفراد | العمالة المنزلية ...
1,2,ماذا أفعل إذا تم إرجاع الطلب للمراجعة؟,الدخول إلى “مهامي” ← اختيار مهمة طلب مراجعة ← ...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
2,3,ماذا يحدث بعد إرسال الطلب؟,يتم تحويل الطلب إلى اللجنة للمراجعة ويتم إشعار...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
3,4,كيف أقدم طلب تعديل فئة مهنية؟,من خلال تسجيل الدخول ← الخدمات ← اختيار “طلب ت...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
4,5,ما متطلبات التقديم؟,أن يكون المتقدم حاصل على ترخيص سابق.,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
5,6,من يمكنه التقديم على خدمة تعديل الفئة المهنية؟,الممارس الحاصل على ترخيص من خلال خدمة التسجيل ...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
6,7,هل يمكن للمتقدم أن يحصل على أكثر من تصنيف وبال...,نعم، يمكن للمتقدم الحصول على أكثر من رخصة إذا ...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
7,8,إذا رفض طلب التسجيل والتصنيف هل يحق لي استرداد...,نعم، يحق لك استرداد رسوم خدمة التسجيل والتصنيف...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
8,9,ماذا يحدث في حال رفض الطلب؟,يمكنك الاطلاع على سبب الرفض من خلال تفاصيل الط...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...
9,10,هل يمكن إلغاء الفاتورة؟,نعم، يمكن إلغاء الفاتورة قبل السداد، مع ضرورة ...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلي...


### 2.1 عدد الأسئلة المسحوبة

In [7]:
print("عدد الأسئلة:", len(df_faq))

display(df_faq[[
    "faq_id",
    "question",
    "beneficiaries",
    "sector",
    "subsite"
]].head(30))

عدد الأسئلة: 711


,faq_id,question,beneficiaries,sector,subsite
0,1,هل تستخدم البوابة والانظمة التابعة لها الحلول ...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية
1,2,ماذا أفعل إذا تم إرجاع الطلب للمراجعة؟,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
2,3,ماذا يحدث بعد إرسال الطلب؟,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
3,4,كيف أقدم طلب تعديل فئة مهنية؟,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
4,5,ما متطلبات التقديم؟,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
5,6,من يمكنه التقديم على خدمة تعديل الفئة المهنية؟,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,,مركز الرياض للسياسات السلوكية
6,7,هل يمكن للمتقدم أن يحصل على أكثر من تصنيف وبال...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
7,8,إذا رفض طلب التسجيل والتصنيف هل يحق لي استرداد...,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
8,9,ماذا يحدث في حال رفض الطلب؟,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,,مركز الرياض للسياسات السلوكية
9,10,هل يمكن إلغاء الفاتورة؟,التخصصات الاجتماعية | العمالة المنزلية | كبار ...,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية


### 2.2 فحص اكتمال السحب

In [11]:
# ============================================================
# فحص اكتمال سحب صفحات FAQ
# مع استثناء الصفحة الأخيرة لأنها قد تحتوي على عدد أقل طبيعيًا
# ============================================================

faq_page_check = (
    df_faq.groupby("page_number")["question"]
    .count()
    .reset_index(name="questions_count")
)

display(faq_page_check)

total_pages = faq_page_check["page_number"].nunique()
last_page = faq_page_check["page_number"].max()

print("عدد الصفحات المسحوبة:", total_pages)
print("إجمالي الأسئلة:", faq_page_check["questions_count"].sum())
print("أقل عدد أسئلة في صفحة:", faq_page_check["questions_count"].min())
print("أعلى عدد أسئلة في صفحة:", faq_page_check["questions_count"].max())

# نراجع فقط الصفحات غير الأخيرة إذا كان عدد الأسئلة فيها قليل جداً
pages_need_review = faq_page_check[
    (faq_page_check["page_number"] != last_page) &
    (faq_page_check["questions_count"] < 10)
]

print("\nصفحات تحتاج مراجعة فعلية:")
display(pages_need_review)

if pages_need_review.empty:
    print("لا توجد صفحات ناقصة بشكل غير طبيعي. الصفحة الأخيرة قد تحتوي على عدد أقل وهذا طبيعي.")

,page_number,questions_count
0,1,12
1,2,11
2,3,12
3,4,12
4,5,12
5,6,12
6,7,12
7,8,12
8,9,12
9,10,12


عدد الصفحات المسحوبة: 60
إجمالي الأسئلة: 711
أقل عدد أسئلة في صفحة: 9
أعلى عدد أسئلة في صفحة: 12

صفحات تحتاج مراجعة فعلية:


,page_number,questions_count


لا توجد صفحات ناقصة بشكل غير طبيعي. الصفحة الأخيرة قد تحتوي على عدد أقل وهذا طبيعي.


### 2.3 معاينة الأسئلة

In [12]:
df_faq[["faq_id", "question", "answer", "page_number"]].head(60)

,faq_id,question,answer,page_number
0,1,هل تستخدم البوابة والانظمة التابعة لها الحلول ...,تقدم الوزارة خدمة التوقيع الالكتروني في اطار ا...,1
1,2,ماذا أفعل إذا تم إرجاع الطلب للمراجعة؟,الدخول إلى “مهامي” ← اختيار مهمة طلب مراجعة ← ...,1
2,3,ماذا يحدث بعد إرسال الطلب؟,يتم تحويل الطلب إلى اللجنة للمراجعة ويتم إشعار...,1
3,4,كيف أقدم طلب تعديل فئة مهنية؟,من خلال تسجيل الدخول ← الخدمات ← اختيار “طلب ت...,1
4,5,ما متطلبات التقديم؟,أن يكون المتقدم حاصل على ترخيص سابق.,1
5,6,من يمكنه التقديم على خدمة تعديل الفئة المهنية؟,الممارس الحاصل على ترخيص من خلال خدمة التسجيل ...,1
6,7,هل يمكن للمتقدم أن يحصل على أكثر من تصنيف وبال...,نعم، يمكن للمتقدم الحصول على أكثر من رخصة إذا ...,1
7,8,إذا رفض طلب التسجيل والتصنيف هل يحق لي استرداد...,نعم، يحق لك استرداد رسوم خدمة التسجيل والتصنيف...,1
8,9,ماذا يحدث في حال رفض الطلب؟,يمكنك الاطلاع على سبب الرفض من خلال تفاصيل الط...,1
9,10,هل يمكن إلغاء الفاتورة؟,نعم، يمكن إلغاء الفاتورة قبل السداد، مع ضرورة ...,1


### 2.4 فحص اكتمال أعمدة الأسئلة

In [13]:
df_faq.count()

faq_id             711
question           711
answer             711
beneficiaries      711
sector             711
subsite            711
page_number        711
page_url           711
source             711
searchable_text    711
dtype: int64

## المرحلة 3: الحفظ النهائي الموحد

هذه الخلية تحفظ مخرجات السحب النظيفة في مكانين:

1. نفس مجلد النوتبوك، لسهولة الفحص المباشر.
2. مجلد المشروع `saudi_labor_law_voice_agent_project/01_raw/`، وهو المسار الذي ستقرأ منه نوتبوكات المعالجة وRAG لاحقاً.

بهذا لا تحتاج لإعادة السحب في كل نوتبوك.


In [14]:
# ============================================================
# Unified Final Save Cell
# حفظ مخرجات السحب النهائية لاستخدامها في بقية النوتبوكات بدون إعادة Scraping
# ============================================================

from pathlib import Path
import json

# ------------------------------------------------------------
# 1) مجلد المشروع المعتمد لبقية النوتبوكات
# ------------------------------------------------------------
PROJECT_DIR = Path.cwd() / "saudi_labor_law_voice_agent_project"
RAW_DIR = PROJECT_DIR / "01_raw"

PROJECT_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2) تجهيز نسخة FAQ نظيفة قبل الحفظ
# ------------------------------------------------------------
df_faq_clean = df_faq.copy()

for col in ["question", "answer", "beneficiaries", "sector", "subsite", "page_url", "source", "searchable_text"]:
    if col in df_faq_clean.columns:
        df_faq_clean[col] = (
            df_faq_clean[col]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

if {"question", "answer"}.issubset(df_faq_clean.columns):
    df_faq_clean = (
        df_faq_clean
        .drop_duplicates(subset=["question", "answer"], keep="first")
        .reset_index(drop=True)
    )

    # إعادة ترقيم faq_id بعد إزالة التكرار
    if "faq_id" in df_faq_clean.columns:
        df_faq_clean = df_faq_clean.drop(columns=["faq_id"])
    df_faq_clean.insert(0, "faq_id", range(1, len(df_faq_clean) + 1))

# ------------------------------------------------------------
# 3) أسماء الملفات المعتمدة
# ------------------------------------------------------------
LAW_OUTPUT_CSV = "saudi_labor_law_by_bab_articles.csv"
LAW_OUTPUT_XLSX = "saudi_labor_law_by_bab_articles.xlsx"
LAW_OUTPUT_JSON = "saudi_labor_law_by_bab_articles.json"

FAQ_OUTPUT_CSV = "hrsd_faq_with_filters_classification.csv"
FAQ_OUTPUT_XLSX = "hrsd_faq_with_filters_classification.xlsx"
FAQ_OUTPUT_JSON = "hrsd_faq_with_filters_classification.json"

# مسارات مجلد المشروع
LAW_PROJECT_CSV = RAW_DIR / LAW_OUTPUT_CSV
LAW_PROJECT_XLSX = RAW_DIR / LAW_OUTPUT_XLSX
LAW_PROJECT_JSON = RAW_DIR / LAW_OUTPUT_JSON

FAQ_PROJECT_CSV = RAW_DIR / FAQ_OUTPUT_CSV
FAQ_PROJECT_XLSX = RAW_DIR / FAQ_OUTPUT_XLSX
FAQ_PROJECT_JSON = RAW_DIR / FAQ_OUTPUT_JSON

# أسماء إضافية واضحة للنسخ النظيفة داخل 01_raw
LAW_CLEAN_CSV = RAW_DIR / "saudi_labor_law_articles_clean.csv"
FAQ_CLEAN_CSV = RAW_DIR / "hrsd_faq_clean.csv"

# ------------------------------------------------------------
# 4) حفظ مواد نظام العمل
# ------------------------------------------------------------
df_labor_law_clean.to_csv(LAW_OUTPUT_CSV, index=False, encoding="utf-8-sig")
df_labor_law_clean.to_excel(LAW_OUTPUT_XLSX, index=False)

with open(LAW_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(df_labor_law_clean.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

# نسخة داخل مجلد المشروع
df_labor_law_clean.to_csv(LAW_PROJECT_CSV, index=False, encoding="utf-8-sig")
df_labor_law_clean.to_excel(LAW_PROJECT_XLSX, index=False)

with open(LAW_PROJECT_JSON, "w", encoding="utf-8") as f:
    json.dump(df_labor_law_clean.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

df_labor_law_clean.to_csv(LAW_CLEAN_CSV, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 5) حفظ FAQ
# ------------------------------------------------------------
df_faq_clean.to_csv(FAQ_OUTPUT_CSV, index=False, encoding="utf-8-sig")
df_faq_clean.to_excel(FAQ_OUTPUT_XLSX, index=False)

with open(FAQ_OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(df_faq_clean.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

# نسخة داخل مجلد المشروع
df_faq_clean.to_csv(FAQ_PROJECT_CSV, index=False, encoding="utf-8-sig")
df_faq_clean.to_excel(FAQ_PROJECT_XLSX, index=False)

with open(FAQ_PROJECT_JSON, "w", encoding="utf-8") as f:
    json.dump(df_faq_clean.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

df_faq_clean.to_csv(FAQ_CLEAN_CSV, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 6) فحص نهائي سريع
# ------------------------------------------------------------
nums = set(df_labor_law_clean["article_number_int"].dropna().astype(int))

print("تم حفظ مخرجات السحب النهائية بنجاح.")
print("\nمجلد المشروع:")
print(RAW_DIR)

print("\nفحص مواد نظام العمل:")
print("عدد الصفوف:", len(df_labor_law_clean))
print("عدد المواد:", len(nums), "| الأعلى:", max(nums) if nums else None)
print("المفقود من 1 إلى 245:", [n for n in range(1, 246) if n not in nums])
print("عدد التكرارات الحقيقية:", df_labor_law_clean["article_number_label"].duplicated().sum())
print("عدد المواد بدون نص:", (df_labor_law_clean["article_text"].astype(str).str.strip() == "").sum())

print("\nتوزيع حالة المواد:")
display(df_labor_law_clean["article_status"].value_counts().to_frame("count"))

print("\nفحص FAQ:")
print("عدد الأسئلة:", len(df_faq_clean))
if {"question", "answer"}.issubset(df_faq_clean.columns):
    print("عدد التكرارات:", df_faq_clean.duplicated(subset=["question", "answer"]).sum())
    print("أسئلة بدون جواب:", (df_faq_clean["answer"].astype(str).str.strip() == "").sum())

print("\nالملفات الأساسية التي ستستخدمها بقية النوتبوكات:")
print(LAW_PROJECT_CSV)
print(FAQ_PROJECT_CSV)


تم حفظ مخرجات السحب النهائية بنجاح.

مجلد المشروع:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\01_raw

فحص مواد نظام العمل:
عدد الصفوف: 249
عدد المواد: 245 | الأعلى: 245
المفقود من 1 إلى 245: []
عدد التكرارات الحقيقية: 0
عدد المواد بدون نص: 0

توزيع حالة المواد:


,count
article_status,
active,211
repealed,38



فحص FAQ:
عدد الأسئلة: 711
عدد التكرارات: 0
أسئلة بدون جواب: 0

الملفات الأساسية التي ستستخدمها بقية النوتبوكات:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\01_raw\saudi_labor_law_by_bab_articles.csv
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\01_raw\hrsd_faq_with_filters_classification.csv


## الخلاصة

بعد تشغيل النوتبوك كاملاً ستكون لديك ملفات جاهزة للاستخدام في بقية المشروع بدون إعادة Scraping:

```text
saudi_labor_law_voice_agent_project/01_raw/saudi_labor_law_by_bab_articles.csv
saudi_labor_law_voice_agent_project/01_raw/hrsd_faq_with_filters_classification.csv
```

المعايير المتوقعة لقبول مرحلة السحب:

- مواد نظام العمل: 245 مادة.
- المفقود من 1 إلى 245: قائمة فارغة.
- التكرارات الحقيقية حسب `article_number_label`: صفر.
- لا توجد مواد بدون نص.
- وجود عمود `article_status` لتمييز المواد الفعالة والملغاة.
- ملف FAQ محفوظ مع إزالة التكرار على أساس السؤال والجواب.

> ملاحظة منهجية: هذه المرحلة مسؤولة فقط عن السحب والحفظ. نوتبوك المعالجة وRAG يجب أن يقرآ ملفات CSV المحفوظة من مجلد `01_raw` بدلاً من إعادة السحب من الموقع.
